# Slug Clustering by Country Coverage

Groups slugs by their country coverage patterns using Jaccard similarity on binary presence matrices.
Missingness is signal — the absence of an indicator for a country is valuable information.

**Phase 1: Model Definition**

In [1]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

╔══════════════════════════════════════════════════════════════════════════╗
║ QoG METADATA JOINING - PHASE 0 LOADED                                ║
╚══════════════════════════════════════════════════════════════════════════╝

Quick Start:
    metadata = join_metadata()              # Run full pipeline (single isomorphism check)
    metadata = join_metadata_with_cascade()  # Run cascade (strictest → loosest), then union on slug
    quick_check()                             # Diagnostic check
    inspect_exceptions()                      # Review configuration
    show_usage()                              # Detailed documentation

Pipeline Steps:
    1. ingest_and_normalize()            # Load & normalize sources (PDF = qog_slugs_temporal.csv; min_year/max_year ingested)
    2. align_id_variables!(...)          # Harmonize ID vars
    3. run_isomorphism_cascade(...)      # Strictest → loosest until success; returns (stata_df, pdf_df, arrow_df) for union on slug
    4. unify_and_join(..

In [ ]:
using CSV, DataFrames

# Load augmented data (uses cached Arrow if available, otherwise runs full pipeline)
df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
geo_df = CSV.read("data/ggis_geographic_lookup.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs, $(nrow(geo_df)) countries")

## Run Clustering Pipeline

Default thresholds — adjust as needed.

In [ ]:
result = run_slug_clustering(df, meta_df, geo_df)

## Cluster Profiles

In [ ]:
result.profiles

## ht_region Alignment

In [ ]:
result.ht_validation

## Explore a Specific Cluster

Change `cid` to inspect different clusters.

In [ ]:
cid = first(result.profiles.cluster_id)
cluster_slugs = filter(r -> r.cluster_id == cid, result.slug_clusters)
println("Cluster $cid: $(nrow(cluster_slugs)) slugs")
leftjoin(cluster_slugs, meta_df[:, [:slug, :prefix, :label, :provenance]], on=:slug)

## Experiment with Thresholds

In [ ]:
# Tighter clusters: higher min_sim, lower k
# result_tight = run_slug_clustering(df, meta_df, geo_df;
#     presence_min_pct=0.15, min_sim=0.20, k=10)

# Looser clusters: lower min_sim, higher k
# result_loose = run_slug_clustering(df, meta_df, geo_df;
#     presence_min_pct=0.05, min_sim=0.05, k=30)

## Save Results

In [ ]:
# CSV.write("data/slug_clusters.csv", result.slug_clusters)
# println("✅ Saved slug_clusters.csv")